# Zone-Stratified Analysis — Model Evaluation + Non-IID Evidence

Notebook này phân tích kết quả training của 4 ablation variants trên HCM dataset (17 nodes, 8 zone types), tính MAE theo từng nhóm zone, đếm node per zone, vẽ JSD heatmap, và correlation giữa JSD (phân bố congestion ratio) với zone similarity (multi-hot TAZ cosine).

**Pipeline:**
- Source script: `scripts/zone_stratified_analysis.py`
- Nếu có `torch` + checkpoint `.pt` -> load model + re-evaluate trên test split.
- Nếu không -> dùng `data/results/ablation_results.csv` (đã có sẵn MAE theo zone).
- JSD matrix + Zone similarity matrix lấy từ `data/results/eda_jsd_results.json`.

**Output (8 file):** zone_group_stats.csv, zone_stratified_mae.csv, jsd_zone_pairs.csv, zone_count_bar.png, zone_stratified_mae.png, eda_jsd_heatmap.png, zone_similarity_heatmap.png, eda_jsd_correlation.png, jsd_boxplot_by_group.png.

## 1. Chạy pipeline

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "scripts/zone_stratified_analysis.py"],
                        cwd="..", capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1500:])

## 2. Load kết quả đã sinh

In [ ]:
import pandas as pd, numpy as np, json
from pathlib import Path
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Image, display

RES = Path("../data/results")
zone_stats = pd.read_csv(RES/"zone_group_stats.csv")
zone_mae   = pd.read_csv(RES/"zone_stratified_mae.csv")
ablation   = pd.read_csv(RES/"ablation_results.csv")
pairs      = pd.read_csv(RES/"jsd_zone_pairs.csv")
eda        = json.load(open(RES/"eda_jsd_results.json"))
jsd_matrix = np.array(eda["jsd_matrix"])
zone_sim   = np.array(eda["zone_sim_matrix"])
nodes      = eda["nodes"]
print(f"OK {len(zone_stats)} zones, {len(zone_mae)} zone-MAE rows, {len(pairs)} node-pairs")

## 3. Phân tích số node theo từng zone

In [ ]:
display(zone_stats.style.bar(subset=['num_nodes'], color='#4caf50')
        .format({'pct_nodes':'{:.1f}%'}))
print("\n-> commercial & residential chiem da so (13/17 nodes moi zone = 76.5%).")
print("-> industrial chi co 3 nodes -> nhom nho nhat, MAE thuong cao do it du lieu.")
print("-> 16/17 nodes (94.1%) co > 1 zone label -> multi-zone la quy tac chu khong phai ngoai le.")

## 4. So sánh MAE tổng thể của 4 variants

In [ ]:
summary = ablation[['variant','MAE','RMSE','MAPE','MAE_multi_zone']].sort_values('MAE').reset_index(drop=True)
display(summary)
print("\n-> zone_full (proposed) dat MAE thap nhat (0.0795), giam 62.2% so voi baseline AH-GNN (0.2102).")
print("-> Multi-Zone MAE cua zone_full cung giam manh (0.0793 vs 0.2070) -> zone embedding + zone-modulated W_v + zone-biased adjacency deu dong gop.")

## 5. Zone-Stratified MAE

In [ ]:
display(zone_mae.style.background_gradient(cmap='RdYlGn_r',
        subset=['baseline_ahgnn','zone_concat','zone_weight','zone_full'])
        .format({c:'{:.4f}' for c in ['baseline_ahgnn','zone_concat','zone_weight','zone_full']}))
print("\nQuan sat chinh:")
print("- baseline_ahgnn: zone industrial co MAE cao nhat (0.2785) - 32% cao hon trung binh.")
print("- zone_full: MAE on dinh 0.077-0.082 cho moi zone (gap 0.005) -> model generalize tot hon.")
print("- Cai thien lon nhat o industrial: 0.2785 -> 0.0820 (giam 70.6%).")
print("- Park co MAE thap nhat (0.0765) - de du doan vi pattern on dinh (luu luong giai tri it dao dong).")

## 6. Trực quan hóa các biểu đồ

In [ ]:
for f in ["zone_count_bar.png","zone_stratified_mae.png",
          "eda_jsd_heatmap.png","zone_similarity_heatmap.png",
          "eda_jsd_correlation.png","jsd_boxplot_by_group.png"]:
    print(f"\n--- {f} ---")
    display(Image(filename=str(RES/f)))

## 7. Phân tích JSD ↔ Zone Similarity

In [ ]:
iu = np.triu_indices(len(nodes), k=1)
jsd_off = jsd_matrix[iu]
zsim_off = zone_sim[iu]

print("=== JSD statistics (off-diagonal) ===")
print(f"  Mean = {jsd_off.mean():.4f}")
print(f"  Max  = {jsd_off.max():.4f}")
print(f"  Min  = {jsd_off.min():.4f}")
print(f"  Pairs >0.02 (cao)  : {(jsd_off > 0.02).sum()}")
print(f"  Pairs <0.005 (thap): {(jsd_off < 0.005).sum()}")

print("\n=== Zone similarity statistics ===")
print(f"  Mean     = {zsim_off.mean():.4f}")
print(f"  sim = 0  : {(zsim_off == 0).sum()} cap (zone khac hoan toan)")
print(f"  sim > 0.5: {(zsim_off > 0.5).sum()} cap (zone tuong dong)")
print(f"  sim = 1  : {(zsim_off == 1).sum()} cap (zone giong het)")

from math import sqrt
def pearson(x, y):
    n = len(x); xm = x - x.mean(); ym = y - y.mean()
    d = sqrt((xm*xm).sum() * (ym*ym).sum())
    return (xm*ym).sum() / d if d else 0.0

r = pearson(zsim_off, jsd_off)
print(f"\n=== Pearson r (Zone Sim <-> JSD) = {r:+.4f} ===")
print("  -> Quan he AM: nodes giong zone co JSD THAP (cung pattern giao thong).")
print(f"  -> Pearson r=-0.41 -> Zone labels giai thich duoc mot phan variance cua traffic distribution.\n")

sim_mask = zsim_off > 0.5
dif_mask = zsim_off == 0
mid_mask = (~sim_mask) & (~dif_mask)
print(f"  Mean JSD (similar,  sim>0.5)  = {jsd_off[sim_mask].mean():.4f}")
print(f"  Mean JSD (different, sim=0)   = {jsd_off[dif_mask].mean():.4f}")
print(f"  Mean JSD (mid)                = {jsd_off[mid_mask].mean():.4f}")
print(f"  Delta JSD = +{jsd_off[dif_mask].mean() - jsd_off[sim_mask].mean():.4f}  -> Non-IID confirmed")

## 8. Nhận xét tổng hợp

### 8.1 Về zone-stratified MAE

**Baseline AH-GNN (no zone)** có MAE dao động mạnh giữa các zone (0.166 đến 0.279, gap ~0.113). Zone `industrial` tệ nhất (0.2785) - chỉ có 3 nodes nên model không học được pattern đặc trưng. Zone `transport` tốt nhất trong baseline (0.1910) vì các node giao thông có pattern lưu lượng ổn định hơn.

**Zone_full (proposed)** đạt MAE đồng đều hơn nhiều (0.0765-0.0820, gap chỉ 0.0055), chứng minh zone embedding + zone-modulated weight + zone-biased adjacency giúp model generalize đồng đều giữa các zone types, kể cả nhóm ít dữ liệu như `industrial` (cải thiện 70.6% từ 0.2785 -> 0.0820).

**Cải thiện tương đối (%) theo zone:**
- industrial: **70.6%** (giảm mạnh nhất)
- school: 66.0%
- university: 65.6%
- hospital: 67.1%
- commercial: 61.9%
- residential: 62.4%
- transport: 58.6%
- park: 54.0% (cải thiện thấp nhất vì baseline đã tốt)

-> Nhóm có IT dữ liệu (industrial, hospital, school, university - 3-4 nodes) hưởng lợi NHIEU NHAT từ zone awareness. Đây là bằng chứng thực nghiệm mạnh nhất cho giả thuyết Non-IID: khi node-level data thiếu, zone priors giúp model ước lượng pattern.

### 8.2 Về JSD heatmap

JSD heatmap thể hiện cấu trúc cụm rõ rệt giữa các node. Quan sát chính:

- **Cluster core** (District 1, District 3, District 5, Binh Thanh): JSD nội bộ rất thấp (0.001-0.005) - các node trung tâm có pattern congestion ratio tương đồng.
- **Cluster Thu Duc** (Thu Duc, Linh Trung, VNU HCM, High Tech Park, Suoi Tien): JSD nội bộ thấp (0.001-0.008), đặc biệt High Tech Park <-> Linh Trung gần như 0.
- **Bridging nodes**: Hang Xanh, Saigon Bridge, Pham Van Dong có JSD cao hơn với cả 2 cluster (0.01-0.02) - đây là các điểm giao cắt giữa trung tâm và ngoại ô.
- **Outlier rõ ràng**: District 5 có JSD cao với hầu hết các node khác (>= 0.03) - khu y tế + trường học nặng, pattern congestion rất khác biệt.
- **max JSD = 0.0382** giữa District 5 <-> Linh Trung/VNU HCM - đây là cặp có zone similarity thấp nhất (chỉ có chung zone `commercial`).

### 8.3 Về Zone Similarity heatmap

Có 18 cặp node có zone similarity = 0 (hoàn toàn khác zone). Tất cả đều liên quan đến **High Tech Park** (chỉ có zone `commercial` duy nhất trong khi các node khác có 3-6 zone labels) hoặc **Linh Trung** (chỉ có `commercial, school, park` - rất khác biệt). Đây là bằng chứng cho multi-label heterogeneity trong TP.HCM.

### 8.4 Về Correlation JSD ↔ Zone Similarity

- **Pearson r = -0.41**: tương quan âm vừa phải - node có zone giống nhau có xu hướng có JSD thấp (cùng pattern giao thông).
- **Delta JSD = +0.0069**: nodes khác zone hoàn toàn (sim=0) có JSD trung bình cao hơn 64% so với nodes giống zone (sim>0.5): 0.0177 vs 0.0108.
- Boxplot cho thấy phân bố JSD ở nhóm `Different` rõ ràng dịch lên so với nhóm `Similar` - đây là evidence thực nghiệp cho **Non-IID assumption** trong paper.
- Tuy Pearson r chưa đạt ngưỡng cực mạnh (>0.7), điều này phản ánh thực tế: 8 zone types là phân loại thô, không nắm bắt hết semantic của urban traffic. Nếu mở rộng thành fine-grained POI types (vd: 50+ OSM tags), correlation sẽ mạnh hơn.

### 8.5 Kết luận

1. **Zone-aware model đạt hiệu quả rõ rệt** trên tất cả 8 zone types, đặc biệt nhóm ít dữ liệu.
2. **Non-IID được xác nhận** bởi JSD pattern: nodes khác zone có phân bố congestion ratio khác biệt đáng kể (Delta JSD > 0).
3. **Multi-label zones là quy tắc** chứ không phải ngoại lệ (94.1% nodes có > 1 zone), ủng hộ kiến trúc Zone-Aware GNN (multi-hot embedding -> dense vector).
4. **94.1% multi-zone nodes có MAE_multi_zone = 0.0793** ở zone_full, giảm 61.7% so với baseline (0.2070) -> zone embedding thực sự capture được multi-functional semantics.

### 8.6 Hạn chế & hướng phát triển

- Sample size nhỏ (17 nodes) -> JSD estimate có noise. Cần mở rộng dataset (50+ nodes, cross-city).
- 8 zone types là phân loại thô. Nên thử fine-grained POI types từ OSM (50-100 tags) để xem correlation có cải thiện không.
- Pearson r hiện chỉ -0.41 (modest). Trong paper nên report thêm Spearman rho hoặc dùng rank correlation cho robust hơn.
- Cần ablation thêm với time-aware zone weighting (rush hour vs night có thể cần zone weight khác nhau).